# Experimento de Manutenção Preditiva Industrial (AI4I 2020)

**Curso:** Tecnologia em Ciência de Dados — Fatec Pompeia Shunji Nishimura  
**Disciplina:** Aplicação de Ciência de Dados (Prof. Dr. João Ricardo Favan)  
**Integrantes:** Hadryan Fortinis de Oliveira Lima, Raul Victor Matsunaga de Carvalho, Washingtom Ferreira Gonçalves

Este notebook reproduz de forma auditável as etapas de pré-processamento, engenharia de atributos físicos, redução de dimensionalidade (Rank vs PCA), comparação de modelos, otimização de hiperparâmetros e validação do artefato exportado para produção.

In [1]:
import json
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, f1_score, average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split

from src.features import RAW_FEATURE_COLUMNS, TARGET_COLUMN, ENGINEERED_FEATURE_COLUMNS, compute_physical_features
from src.train_pipeline import compute_eda_summary, compute_rank_and_pca_diagnostics

df = pd.read_csv(ROOT / "data" / "ai4i2020.csv", encoding="utf-8-sig")
print(f"Formato do dataset: {df.shape}")
df.head(5)

Formato do dataset: (10000, 14)


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


## 1. Auditoria Exploratória e Prevenção de Vazamento de Alvo

Removemos identificadores (`UDI`, `Product ID`) e os indicadores binários específicos de submodo de falha (`TWF`, `HDF`, `PWF`, `OSF`, `RNF`), que constituiriam vazamento direto da variável-alvo `Machine failure`.

In [2]:
eda = compute_eda_summary(df)
print("Ausentes totais:", eda["missing_values_total"], "| Duplicados:", eda["duplicate_rows"])
print("Distribuição da classe alvo:", eda["target_distribution"])
print("Submodos de falha:", eda["failure_submodes"])
print("Anomalias documentadas:", eda["anomalies_documented"])

Ausentes totais: 0 | Duplicados: 0
Distribuição da classe alvo: {'normal_0': 9661, 'failure_1': 339, 'failure_rate_pct': 3.39}
Submodos de falha: {'TWF': 46, 'HDF': 115, 'PWF': 95, 'OSF': 98, 'RNF': 19}
Anomalias documentadas: {'rnf_1_with_machine_failure_0': 18, 'machine_failure_1_without_submode': 9}


## 2. Divisão Estratificada e Engenharia de Atributos Físicos

Aplicamos divisão estratificada 80/20 (`random_state=42`) e criamos 4 atributos com fundamentação termodinâmica e mecânica: `Temp_Diff_K`, `Mechanical_Power_W`, `Wear_Torque_Nm_min` e `Torque_Speed_Ratio`.

In [3]:
X = df[RAW_FEATURE_COLUMNS].copy()
y = df[TARGET_COLUMN].astype(int).copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
X_train_fe = compute_physical_features(X_train)
display(X_train_fe[ENGINEERED_FEATURE_COLUMNS].describe().round(3))

,Temp_Diff_K,Mechanical_Power_W,Wear_Torque_Nm_min,Torque_Speed_Ratio
count,8000.000,8000.000,8000.000,8000.000
mean,10.001,6282.618,4302.684,0.027
std,1.000,1072.419,2820.177,0.009
min,7.600,1148.441,0.000,0.001
25%,9.300,5559.606,1953.325,0.021
50%,9.800,6272.294,3998.700,0.027
75%,11.000,7013.867,6265.050,0.033
max,12.100,10469.923,16497.000,0.064


## 3. Redução de Dimensionalidade: Rank (ANOVA F / MI) vs PCA

In [4]:
diag = compute_rank_and_pca_diagnostics(X_train, y_train)
rank_df = pd.DataFrame(diag["feature_ranking"])
display(rank_df[["anova_rank", "feature", "is_engineered", "anova_f", "mutual_info", "selected_anova_k10"]])
print(f"PCA para 95% de variância acumulada requer {diag['pca']['n_components_95_var']} componentes principais.")

,anova_rank,feature,is_engineered,anova_f,mutual_info,selected_anova_k10
0,1,Torque_Speed_Ratio,True,349.716,0.04707,True
1,2,Torque [Nm],False,299.653,0.04529,True
2,3,Wear_Torque_Nm_min,True,285.822,0.03340,True
3,4,Mechanical_Power_W,True,255.657,0.04659,True
4,5,Temp_Diff_K,True,100.344,0.00941,True
5,6,Tool wear [min],False,79.510,0.01064,True
6,7,Air temperature [K],False,59.595,0.00740,True
7,8,Type_L,False,17.828,0.00000,True
8,9,Process temperature [K],False,13.368,0.00393,True
9,10,Rotational speed [rpm],False,12.796,0.03612,True


PCA para 95% de variância acumulada requer 5 componentes principais.


## 4. Resultados Consolidados do Experimento (Ablação, Modelos e Otimização)

In [5]:
metrics_report = json.loads((ROOT / "reports" / "metricas_experimento.json").read_text(encoding="utf-8"))

print("=== Ablação (Raw vs FE vs Rank vs PCA) ===")
display(pd.DataFrame(metrics_report["feature_engineering"]["ablation_study"]))

print("=== Comparativo dos 6 Algoritmos Padrão (CV 5-fold e Teste) ===")
comp_cols = ["model_name", "cv_f1", "cv_f1_std", "test_f1", "test_precision", "test_recall", "test_roc_auc", "test_pr_auc"]
display(pd.DataFrame(metrics_report["default_model_comparison"])[comp_cols])

print("=== Comparação Padrão vs Otimizado ===")
opt = metrics_report["optimization"]
print("Modelo base:", opt["selected_base_model"])
print("Melhores hiperparâmetros:", opt["best_hyperparameters"])
print("Padrão (Teste):", opt["default_test"])
print("Otimizado (Teste):", opt["tuned_test"])

=== Ablação (Raw vs FE vs Rank vs PCA) ===


,regime,regime_short,use_engineered,reduction,rf_cv_f1,gb_cv_f1,hgb_cv_f1,rf_cv_pr_auc,gb_cv_pr_auc,hgb_cv_pr_auc
0,"1. Dados Brutos (Sem FE, Sem Redução)",Brutos\n(6 orig.),False,none,0.5946,0.7090,0.7375,0.7638,0.7839,0.8045
1,2. FE + Sem Redução (12 atribs. codif.),FE Completo\n(12 atribs.),True,none,0.8769,0.8902,0.8518,0.8926,0.8902,0.8853
2,3. FE + Rank ANOVA (k = 10 atribs.),FE + Rank\nANOVA (k=10),True,anova_k10,0.8703,0.8874,0.8599,0.8901,0.9099,0.8787
3,4. FE + Rank MI (k = 10 atribs.),FE + Rank\nMI (k=10),True,mi_k10,0.8690,0.8787,0.8462,0.8872,0.8717,0.8754
4,5. FE + PCA 95% (5 comp.),FE + PCA 95%\n(5 comp.),True,pca_95,0.5839,0.5412,0.5735,0.6557,0.6363,0.6629


=== Comparativo dos 6 Algoritmos Padrão (CV 5-fold e Teste) ===


,model_name,cv_f1,cv_f1_std,test_f1,test_precision,test_recall,test_roc_auc,test_pr_auc
0,Dummy (Estratificado),0.0395,0.0070,0.0286,0.0278,0.0294,0.4966,0.0338
1,Regressão Logística,0.3528,0.0996,0.2697,0.5714,0.1765,0.9279,0.4511
2,Rede Neural MLP,0.5305,0.0843,0.6168,0.8462,0.4853,0.9662,0.7021
3,Random Forest,0.8703,0.0295,0.8800,0.9649,0.8088,0.9753,0.8885
4,Gradient Boosting,0.8874,0.0227,0.8943,1.0000,0.8088,0.9716,0.9041
5,HistGradientBoosting,0.8599,0.0149,0.8819,0.9492,0.8235,0.9843,0.9012


=== Comparação Padrão vs Otimizado ===
Modelo base: HistGradientBoosting
Melhores hiperparâmetros: {'class_weight': 'balanced', 'l2_regularization': 0.0, 'learning_rate': 0.1, 'max_iter': 350, 'max_leaf_nodes': 31, 'min_samples_leaf': 10}
Padrão (Teste): {'test_accuracy': 0.9925, 'test_precision': 0.9492, 'test_recall': 0.8235, 'test_f1': 0.8819, 'test_roc_auc': 0.9843, 'test_pr_auc': 0.9012, 'confusion_matrix': {'tn': 1929, 'fp': 3, 'fn': 12, 'tp': 56}}
Otimizado (Teste): {'test_accuracy': 0.994, 'test_precision': 0.9828, 'test_recall': 0.8382, 'test_f1': 0.9048, 'test_roc_auc': 0.9716, 'test_pr_auc': 0.8866, 'confusion_matrix': {'tn': 1931, 'fp': 1, 'fn': 11, 'tp': 57}}


## 5. Validação do Pipeline Exportado para Produção

In [6]:
prod_pipeline = joblib.load(ROOT / "models" / "modelo_manutencao_otimizado.joblib")
y_pred = prod_pipeline.predict(X_test)
y_prob = prod_pipeline.predict_proba(X_test)[:, 1]

print(f"F1 (Teste): {f1_score(y_test, y_pred):.4f}")
print(f"ROC-AUC (Teste): {roc_auc_score(y_test, y_prob):.4f}")
print(f"PR-AUC (Teste): {average_precision_score(y_test, y_prob):.4f}")
print("Matriz de Confusão:\n", confusion_matrix(y_test, y_pred))
print("\nRelatório de Classificação:\n", classification_report(y_test, y_pred, digits=4))

F1 (Teste): 0.9048
ROC-AUC (Teste): 0.9716
PR-AUC (Teste): 0.8866
Matriz de Confusão:
 [[1931    1]
 [  11   57]]

Relatório de Classificação:
               precision    recall  f1-score   support

           0     0.9943    0.9995    0.9969      1932
           1     0.9828    0.8382    0.9048        68

    accuracy                         0.9940      2000
   macro avg     0.9885    0.9189    0.9508      2000
weighted avg     0.9939    0.9940    0.9938      2000

